In [1]:
# Import required libraries
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.nn.utils.rnn import pad_sequence

# Re-define the LSTM model class (same architecture as before, needed to load the saved weights)
class SepsisLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        lstm_out = self.dropout(lstm_out)
        out = torch.sigmoid(self.fc(lstm_out))
        return out.squeeze(-1)

# Create a model instance with the same structure, then load the saved trained weights
model = SepsisLSTM(n_features=71, hidden_size=64)
model.load_state_dict(torch.load("../models/lstm_model.pt"))

print("Model loaded successfully")

Model loaded successfully


In [2]:
# Load validation data
val_df = pd.read_parquet("../data/processed/val_final.parquet")

# Feature columns (same as before)
exclude_cols = ["patient_id", "SepsisLabel"]
feature_cols = [c for c in val_df.columns if c not in exclude_cols]

# Function to create sequences (same as training time)
def create_sequences(data, feature_cols, max_len=100):
    sequences = []
    labels = []
    for pid in data["patient_id"].unique():
        patient_data = data[data["patient_id"] == pid].sort_values("ICULOS")
        seq = patient_data[feature_cols].values
        label = patient_data["SepsisLabel"].values
        if len(seq) > max_len:
            seq = seq[:max_len]
            label = label[:max_len]
        sequences.append(seq)
        labels.append(label)
    return sequences, labels

# Function to pad sequences
def prepare_padded_data(sequences, labels):
    seq_tensors = [torch.tensor(seq, dtype=torch.float32) for seq in sequences]
    label_tensors = [torch.tensor(lab, dtype=torch.float32) for lab in labels]
    lengths = [len(seq) for seq in sequences]
    padded_seqs = pad_sequence(seq_tensors, batch_first=True, padding_value=0)
    padded_labels = pad_sequence(label_tensors, batch_first=True, padding_value=0)
    return padded_seqs, padded_labels, lengths

# Create sequences for validation set
val_sequences, val_labels = create_sequences(val_df, feature_cols)
val_padded_seqs, val_padded_labels, val_lengths = prepare_padded_data(val_sequences, val_labels)

print("Validation sequences ready:", val_padded_seqs.shape)

Validation sequences ready: torch.Size([6050, 100, 71])


In [3]:
from sklearn.preprocessing import StandardScaler

# Load training data again to fit the scaler (scaler must be fit on training data, same as before)
train_df = pd.read_parquet("../data/processed/train_final.parquet")
train_sequences, train_labels = create_sequences(train_df, feature_cols)
train_padded_seqs, train_padded_labels, train_lengths = prepare_padded_data(train_sequences, train_labels)

# Fit scaler on training data
scaler = StandardScaler()
n_patients_train, max_len, n_feat = train_padded_seqs.shape
train_flat = train_padded_seqs.numpy().reshape(-1, n_feat)
scaler.fit(train_flat)

# Apply the same scaler to validation data
n_patients_val = val_padded_seqs.shape[0]
val_flat = val_padded_seqs.numpy().reshape(-1, n_feat)
val_scaled = scaler.transform(val_flat).reshape(n_patients_val, max_len, n_feat)
val_padded_seqs_scaled = torch.tensor(val_scaled, dtype=torch.float32)

print("Scaling done, shape:", val_padded_seqs_scaled.shape)

Scaling done, shape: torch.Size([6050, 100, 71])


In [4]:
# Function to get MC Dropout predictions - keeps dropout ACTIVE during prediction
def mc_dropout_predict(model, x, n_samples=30):
    # Keep model in "train" mode so dropout stays active (this is the key trick)
    model.train()
    
    predictions = []
    
    # Make n_samples predictions, each with different random dropout patterns
    with torch.no_grad():
        for _ in range(n_samples):
            pred = model(x)
            predictions.append(pred.unsqueeze(0))
    
    # Stack all predictions together
    all_preds = torch.cat(predictions, dim=0)
    
    # Mean = the average risk prediction (our final risk score)
    mean_pred = all_preds.mean(dim=0)
    
    # Standard deviation = how much predictions varied (our uncertainty measure)
    std_pred = all_preds.std(dim=0)
    
    return mean_pred, std_pred

print("MC Dropout function ready")

MC Dropout function ready


In [5]:
# Run MC Dropout prediction on validation data (this will take a bit longer since we predict 30 times)
mean_predictions, uncertainty_scores = mc_dropout_predict(model, val_padded_seqs_scaled, n_samples=30)

print("Mean predictions shape:", mean_predictions.shape)
print("Uncertainty scores shape:", uncertainty_scores.shape)

# Look at a few examples - patient risk score along with uncertainty
print("\nSample results (first patient, first 5 hours):")
print("Mean risk:", mean_predictions[0, :5])
print("Uncertainty:", uncertainty_scores[0, :5])

Mean predictions shape: torch.Size([6050, 100])
Uncertainty scores shape: torch.Size([6050, 100])

Sample results (first patient, first 5 hours):
Mean risk: tensor([0.0422, 0.0337, 0.0512, 0.0518, 0.0468])
Uncertainty: tensor([0.0144, 0.0123, 0.0156, 0.0170, 0.0135])


In [6]:
# Extract only real (non-padded) predictions and uncertainties, along with true labels
all_mean_preds = []
all_uncertainties = []
all_true_labels = []

for i in range(len(val_lengths)):
    real_len = val_lengths[i]
    all_mean_preds.extend(mean_predictions[i, :real_len].numpy())
    all_uncertainties.extend(uncertainty_scores[i, :real_len].numpy())
    all_true_labels.extend(val_padded_labels[i, :real_len].numpy())

all_mean_preds = np.array(all_mean_preds)
all_uncertainties = np.array(all_uncertainties)
all_true_labels = np.array(all_true_labels)

# Check average uncertainty for correct vs high-risk predictions
print("Average uncertainty for LOW risk predictions (<0.3):", 
      all_uncertainties[all_mean_preds < 0.3].mean())
print("Average uncertainty for HIGH risk predictions (>=0.3):", 
      all_uncertainties[all_mean_preds >= 0.3].mean())

# Check if uncertainty correlates with actual sepsis cases
print("\nAverage uncertainty for actual NON-sepsis hours:", 
      all_uncertainties[all_true_labels == 0].mean())
print("Average uncertainty for actual SEPSIS hours:", 
      all_uncertainties[all_true_labels == 1].mean())

Average uncertainty for LOW risk predictions (<0.3): 0.0057537984
Average uncertainty for HIGH risk predictions (>=0.3): 0.041286748

Average uncertainty for actual NON-sepsis hours: 0.0056402246
Average uncertainty for actual SEPSIS hours: 0.013534244


In [7]:
import json

uncertainty_results = {
    "avg_uncertainty_low_risk": float(all_uncertainties[all_mean_preds < 0.3].mean()),
    "avg_uncertainty_high_risk": float(all_uncertainties[all_mean_preds >= 0.3].mean()),
    "avg_uncertainty_non_sepsis": float(all_uncertainties[all_true_labels == 0].mean()),
    "avg_uncertainty_sepsis": float(all_uncertainties[all_true_labels == 1].mean())
}

with open("../reports/uncertainty_results.json", "w") as f:
    json.dump(uncertainty_results, f, indent=2)

# Also save the raw predictions and uncertainties for later use (e.g., in the demo)
np.save("../reports/mean_predictions.npy", all_mean_preds)
np.save("../reports/uncertainty_scores.npy", all_uncertainties)

print("Uncertainty results saved!")

Uncertainty results saved!
